# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a reproducible walkthrough for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. All record sets, fields, and columns are referenced by their `@id` attributes according to the Croissant schema.

### Dataset Source
The dataset is described by the following Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(metadata.name + ': ' + metadata.description)
print('Identifier:', getattr(metadata, 'identifier', None))
print('Version: ', getattr(metadata, 'version', None))


## 2. Data Overview
Review available record sets and their fields, referenced by their `@id`s.

Below, we enumerate the record sets (`@id`) in the dataset, along with the fields (columns) in each.

In [ ]:
# List all Record Sets and their fields using their @id

recordsets = dataset.record_sets
recordset_ids = []
for rs in recordsets:
    print(f"Record Set: {rs['@id']} | Name: {rs.get('name', '')}")
    recordset_ids.append(rs['@id'])
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields (by @id):")
    for fld in fields:
        if isinstance(fld, str):
            # Only an @id is given
            print(f"    - {fld}")
        elif isinstance(fld, dict):
            # Full field description
            print(f"    - {fld.get('@id', '')} ({fld.get('name', '')})")
    print("")

## 3. Data Extraction
Load data from the record set(s) of interest into pandas DataFrames for analysis. We use the record set `@id`s and field `@id`s identified above. For this dataset, we will extract every record set.

In [ ]:
# Extract all available record sets by @id

all_dataframes = {}
for rs_id in recordset_ids:
    try:
        records_iter = dataset.records(record_set=rs_id)
        records_list = list(records_iter)
        df = pd.DataFrame(records_list)
        all_dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set {rs_id}, shape: {df.shape}")
        print(f"Columns (@id): {list(df.columns)}\n")
    except Exception as e:
        print(f"Failed to load record set {rs_id}: {e}\n")

# For demonstration, pick the first available record set
if len(all_dataframes) > 0:
    example_rs_id = list(all_dataframes.keys())[0]
    print(f"Example DataFrame columns for {example_rs_id}:")
    print(all_dataframes[example_rs_id].columns.tolist())
    display(all_dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We now perform example EDA operations using field and record set `@id`s. Replace the example field and record set `@id`s with those of interest from the overview above.

**Steps:**
- Filter records based on a numeric field (e.g., Age > 60)
- Normalize a numeric field
- Group by a categorical field (e.g., Sex)

Ensure your selected field is numeric for filtering/normalization, and select a plausible group field.

In [ ]:
# Example: adjust the record set and field @id's as found in previous cell
# Here, we demonstrate using the first available record set

record_set_id = example_rs_id
df = all_dataframes[record_set_id]

# Let's automatically select fields that appear numeric
numeric_field = None
for col in df.columns:
    try:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
        # Try coercing the first 10 values to float
        pd.to_numeric(df[col].dropna().iloc[:10])
        numeric_field = col
        break
    except:
        continue

if numeric_field is None:
    print("No numeric field found. Please select a numeric @id column from above.")
else:
    # Try to convert to numeric just in case
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].dropna().quantile(0.7)  # e.g., top 30% for demonstration
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    mean = filtered_df[numeric_field].mean()
    std = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Search for a categorical field to group by (typically of type object, few unique values, e.g. sex/location)
    group_field = None
    for col in df.columns:
        if col == numeric_field:
            continue
        if df[col].nunique() > 1 and df[col].nunique() < 10 and df[col].dtype == object:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

- Example: plot the distribution of a numeric field (e.g., Age)
- Example: boxplot of numeric field by a grouping field (e.g., Sex)

All plots use columns referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} (by @id)")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

if group_field and numeric_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"Boxplot of {numeric_field} by {group_field} (by @id)")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

We have successfully loaded and explored the FAIR^2 colorectal cancer survivors dataset using its Croissant schema via `mlcroissant`. Key steps included:
- Accessing and inspecting dataset metadata
- Enumerating and extracting all record sets and fields by their `@id`
- Performing sample filtering and grouping using field `@id`s
- Visualizing distributions for numeric and categorical variables

All dataset elements, including record sets and fields, were referenced strictly by their Croissant `@id` values. This process demonstrates best practices for interoperable, FAIR-compliant data analysis using the Croissant metadata standard and supporting library.